# Linear Search

Objetivo: Percorrer sequencialmente cada elemento de uma colecao ate encontrar o alvo ou confirmar sua ausencia.

Complexidade:
- Tempo: O(n) no pior caso, O(1) no melhor caso
- Espaco: O(1)

Categoria: Busca

## Diagrama do fluxo

```mermaid
flowchart TD
    A[Inicio] --> B[i = 0]
    B --> C{i < len arr ?}
    C -->|Nao| D[Retornar -1 nao encontrado]
    C -->|Sim| E{arr i == alvo ?}
    E -->|Sim| F[Retornar i posicao]
    E -->|Nao| G[i = i + 1]
    G --> C
```

## Fundamentos

Linear search e o algoritmo de busca mais fundamental da computacao. Sua logica e direta: examine cada elemento da colecao, um a um, ate encontrar o que procura ou esgotar todos os candidatos.

Apesar de sua simplicidade aparente, o linear search e a unica opcao viavel quando os dados nao estao ordenados, quando a colecao e pequena, ou quando o custo de ordenar ou indexar supera o custo das buscas.

Em cenarios reais como analise de logs de servidor, streams de dados em tempo real e filtros sobre conjuntos de dados arbitrarios, o linear search e frequentemente a escolha correta.

In [ ]:
def linear_search(collection, target):
    """
    Searches for target in collection using linear scan.

    Args:
        collection: Iterable of comparable elements.
        target: Value to search for.

    Returns:
        Index of first occurrence, or -1 if not found.

    Time:  O(n) worst case, O(1) best case
    Space: O(1)
    """
    for index, element in enumerate(collection):
        if element == target:
            return index
    return -1


# Basic tests
data = [42, 17, 93, 8, 55, 31, 77, 4]
print(f"Dataset:          {data}")
print(f"Search for 55:    index {linear_search(data, 55)}")   # 4
print(f"Search for 4:     index {linear_search(data, 4)}")    # 7
print(f"Search for 999:   index {linear_search(data, 999)}")  # -1

## Visualizacao passo a passo

O trecho abaixo instrumenta o algoritmo para exibir cada comparacao realizada, permitindo observar o comportamento interno antes de qualquer otimizacao.

In [ ]:
def linear_search_verbose(collection, target):
    """
    Linear search with step-by-step output for teaching purposes.
    Returns index found or -1, and total comparisons made.
    """
    comparisons = 0
    print(f"Searching for '{target}' in {list(collection)}")
    print("-" * 60)

    for index, element in enumerate(collection):
        comparisons += 1
        match = element == target
        status = "FOUND" if match else "skip"
        print(f"  Step {comparisons:>2}: index={index}  value={element!r:<12}  -> {status}")
        if match:
            print(f"\nResult: found at index {index} after {comparisons} comparison(s).")
            return index, comparisons

    print(f"\nResult: not found after {comparisons} comparison(s).")
    return -1, comparisons


log_levels = ["INFO", "DEBUG", "INFO", "WARNING", "ERROR", "INFO", "DEBUG"]
linear_search_verbose(log_levels, "ERROR")

## Otimizacao: Sentinel Search

A cada iteracao, o loop padrao realiza **duas verificacoes**: o teste de limite `i < n` e a comparacao com o alvo. A tecnica do sentinel elimina a verificacao de limite inserindo o proprio alvo no final da colecao. O loop termina sempre ao encontrar o sentinel; depois verificamos se chegamos por um match real ou apenas pelo sentinel.

Essa otimizacao reduz o numero de operacoes por iteracao de 2 para 1, o que pode ser significativo em colecoes com dezenas de milhoes de elementos.

In [ ]:
def sentinel_search(collection, target):
    """
    Linear search with sentinel optimization.
    Eliminates the bounds check inside the loop by appending
    the target as a sentinel at the end of a temporary copy.

    Args:
        collection: List of comparable elements.
        target: Value to search for.

    Returns:
        Index of first occurrence, or -1 if not found.

    Time:  O(n)
    Space: O(n)  -- due to the copy; O(1) if mutating in-place is acceptable
    """
    n = len(collection)
    # Append sentinel so the loop always terminates naturally
    arr = list(collection) + [target]

    i = 0
    while arr[i] != target:  # only one check per iteration
        i += 1

    # If we stopped before the sentinel position, it's a real match
    return i if i < n else -1


data = [10, 30, 50, 70, 90, 110]
print(f"sentinel_search(data, 70):  {sentinel_search(data, 70)}")   # 3
print(f"sentinel_search(data, 999): {sentinel_search(data, 999)}")  # -1

## Caso real: buscando padroes de erro em logs de servidor

Logs de servidor chegam em ordem cronologica e nao estao ordenados por severity. O linear search e a abordagem natural para filtrar entradas que correspondem a um padrao de erro especifico, especialmente quando o log e consumido como stream e nao cabe inteiro na memoria.

In [ ]:
import re
from dataclasses import dataclass
from typing import Iterator


@dataclass
class LogEntry:
    timestamp: str
    level: str
    service: str
    message: str

    def __repr__(self) -> str:
        return f"[{self.timestamp}] {self.level:<7} {self.service}: {self.message}"


SAMPLE_LOGS = [
    LogEntry("2024-03-01 08:01:12", "INFO",    "api-gateway",  "Request received: POST /payments"),
    LogEntry("2024-03-01 08:01:12", "DEBUG",   "auth-service", "Token validated for user_id=7821"),
    LogEntry("2024-03-01 08:01:13", "INFO",    "payment-svc",  "Processing transaction txn_id=TXN-9910"),
    LogEntry("2024-03-01 08:01:14", "ERROR",   "payment-svc",  "DB connection timeout after 5000ms: pool exhausted"),
    LogEntry("2024-03-01 08:01:14", "WARNING", "payment-svc",  "Retry 1/3 for txn_id=TXN-9910"),
    LogEntry("2024-03-01 08:01:15", "ERROR",   "payment-svc",  "DB connection timeout after 5000ms: pool exhausted"),
    LogEntry("2024-03-01 08:01:15", "WARNING", "payment-svc",  "Retry 2/3 for txn_id=TXN-9910"),
    LogEntry("2024-03-01 08:01:16", "INFO",    "metrics-svc",  "Health check OK"),
    LogEntry("2024-03-01 08:01:17", "ERROR",   "payment-svc",  "Max retries exceeded. txn_id=TXN-9910 failed"),
    LogEntry("2024-03-01 08:01:17", "INFO",    "api-gateway",  "Response 503 sent to client"),
]


def find_first_error(logs: list[LogEntry]) -> int:
    """Returns index of first ERROR entry, or -1 if none found."""
    return linear_search([e.level for e in logs], "ERROR")


def search_all_by_pattern(logs: list[LogEntry], pattern: str) -> list[int]:
    """
    Returns indices of all entries whose message matches `pattern`.
    Demonstrates search-all variant of linear search.
    """
    regex = re.compile(pattern, re.IGNORECASE)
    return [i for i, entry in enumerate(logs) if regex.search(entry.message)]


# --- Demo ---
first_err = find_first_error(SAMPLE_LOGS)
print(f"First ERROR at index {first_err}: {SAMPLE_LOGS[first_err]}")

print()
timeout_indices = search_all_by_pattern(SAMPLE_LOGS, r"timeout")
print(f"Entries matching 'timeout' (indices {timeout_indices}):")
for i in timeout_indices:
    print(f"  {SAMPLE_LOGS[i]}")

## Analise de Complexidade

### Tempo

**Melhor caso: O(1)** - O alvo e o primeiro elemento.

**Caso medio: O(n/2) = O(n)** - Em media, percorremos metade da colecao antes de encontrar o alvo.

**Pior caso: O(n)** - O alvo esta no ultimo elemento ou nao existe na colecao.

### Espaco

**O(1)** - Apenas variaveis de controle. A variante sentinel requer O(n) se fizer uma copia, mas pode operar O(1) mutando a colecao original.

### Quando linear search e a escolha certa

| Cenario | Justificativa |
|---|---|
| Dados nao ordenados | Ordenar custa O(n log n); busca unica nao compensa |
| Colecao pequena (< ~50 elementos) | Overhead de estruturas complexas supera o ganho |
| Stream / dados chegando em tempo real | Nao e possivel ordenar sem acumular todo o stream |
| Busca por predicado complexo | Indices nao cobrem logica arbitraria |
| Busca de todas as ocorrencias | Algoritmos de busca binaria encontram apenas uma |

## Comparacao com alternativas nativas do Python

O Python oferece `in` e `.index()` como alternativas para busca linear. Ambos percorrem a lista elemento a elemento internamente (CPython em C), o que os torna mais rapidos na pratica por evitar o overhead do interpretador Python.

In [ ]:
import timeit
import random

SIZES = [1_000, 10_000, 100_000]
RUNS = 200

print(f"{'n':>10} | {'linear_search (ms)':>20} | {'in operator (ms)':>18} | {'.index() (ms)':>15}")
print("-" * 70)

for n in SIZES:
    data = list(range(n))  # ordered so worst case = last element
    target = n - 1         # worst case: always at the end

    t_linear = timeit.timeit(
        lambda d=data, t=target: linear_search(d, t),
        number=RUNS
    ) / RUNS * 1000

    t_in = timeit.timeit(
        lambda d=data, t=target: t in d,
        number=RUNS
    ) / RUNS * 1000

    t_index = timeit.timeit(
        lambda d=data, t=target: d.index(t),
        number=RUNS
    ) / RUNS * 1000

    print(f"{n:>10} | {t_linear:>20.4f} | {t_in:>18.4f} | {t_index:>15.4f}")

print()
print("Nota: 'in' e '.index()' sao implementados em C no CPython, por isso")
print("sao mais rapidos que o linear_search escrito em Python puro.")
print("A complexidade assintotica e identica: O(n).")

## Exercicios

**Desafio 1:** Implemente `search_all(collection, target)` que retorna uma lista com os indices de **todas** as ocorrencias do alvo (nao apenas a primeira). Aplique a busca para encontrar todos os registros de um servico especifico nos logs de exemplo.

**Desafio 2:** Implemente `linear_search_sorted_early_exit(collection, target)` que realiza busca linear em uma lista **ja ordenada** e interrompe a varredura assim que encontrar um elemento maior que o alvo (terminacao antecipada). Compare o numero de comparacoes com a versao sem terminacao antecipada para um alvo inexistente.

In [ ]:
# Desafio 1: search_all - todas as ocorrencias
def search_all(collection, target):
    """
    Returns a list of indices for every occurrence of target.
    Time: O(n) -- must visit every element to guarantee completeness.
    """
    return [i for i, element in enumerate(collection) if element == target]


levels = [e.level for e in SAMPLE_LOGS]
error_indices = search_all(levels, "ERROR")
print(f"All ERROR entries at indices {error_indices}:")
for i in error_indices:
    print(f"  {SAMPLE_LOGS[i]}")

print()

# Desafio 2: terminacao antecipada em dados ordenados
def linear_search_sorted_early_exit(collection, target):
    """
    Linear search on a sorted collection with early exit.
    Stops as soon as an element greater than target is found.
    Returns (index_or_minus1, comparisons_count).
    """
    comparisons = 0
    for i, element in enumerate(collection):
        comparisons += 1
        if element == target:
            return i, comparisons
        if element > target:  # sorted: target cannot appear further right
            break
    return -1, comparisons


sorted_data = list(range(0, 200, 2))  # [0, 2, 4, ..., 198]
missing_target = 75                   # odd number, not in list

idx_plain, cmp_plain = linear_search_verbose(sorted_data, missing_target) if False else (
    linear_search(sorted_data, missing_target), len(sorted_data)
)
idx_early, cmp_early = linear_search_sorted_early_exit(sorted_data, missing_target)

print(f"Sorted list: 0, 2, 4, ... 198  (n={len(sorted_data)})")
print(f"Target: {missing_target} (odd, not in list)")
print(f"Standard linear:    {cmp_plain} comparisons")
print(f"Early-exit sorted:  {cmp_early} comparisons")

## Referencias

1. Knuth, D. E. "The Art of Computer Programming, Vol. 3: Sorting and Searching" (2nd ed.). Addison-Wesley, 1998, pp. 396-408.
2. Cormen, T. H., et al. "Introduction to Algorithms" (3rd ed.). MIT Press, 2009, pp. 148-150.
3. Sedgewick, R., & Wayne, K. "Algorithms" (4th ed.). Addison-Wesley, 2011, pp. 48-50.